# HYZ 2026 Public Colab

Private `HYZ_2026` ana reposunu submodule'lar ile klonlar, ortamı kurar ve çalıştırır.


## 1. Colab Secrets

Şunları ekleyin:

- `GITHUB_TOKEN`
- `TEAM_NAME`
- `PASSWORD`
- `EVALUATION_SERVER_URL`

Token sadece clone ortamına verilir.


In [ ]:
import os
import shutil
import stat
import subprocess
from pathlib import Path
from google.colab import userdata

REPOSITORY_URL = "https://github.com/RaidynTeam/HYZ_2026.git"
REPOSITORY_BRANCH = "main"
REPOSITORY_DIR = Path("/content/HYZ_2026")

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secrets içinde GITHUB_TOKEN eksik")

os.chdir("/content")
shutil.rmtree(REPOSITORY_DIR, ignore_errors=True)
askpass = Path("/content/.hyz_git_askpass.sh")
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
clone_environment = os.environ.copy()
clone_environment.update({
    "GITHUB_TOKEN": github_token,
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
})
try:
    subprocess.run(
        [
            "git", "clone", "--recurse-submodules",
            "--branch", REPOSITORY_BRANCH, "--single-branch",
            REPOSITORY_URL, str(REPOSITORY_DIR),
        ],
        check=True,
        env=clone_environment,
    )
    subprocess.run(
        ["git", "submodule", "update", "--init", "--recursive"],
        check=True,
        cwd=REPOSITORY_DIR,
        env=clone_environment,
    )
finally:
    askpass.unlink(missing_ok=True)
    clone_environment.pop("GITHUB_TOKEN", None)
    del github_token

os.chdir(REPOSITORY_DIR)
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()
submodules = subprocess.check_output(
    ["git", "submodule", "status"], text=True
).strip()
print(f"Repository hazır: branch={REPOSITORY_BRANCH} commit={commit}")
print(submodules)


## 2. Python 3.12 ve bağımlılıklar

Ana repo ve submodule'lar aynı ortamda kurulur.


In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv"],
    check=True,
)
subprocess.run(["uv", "python", "install", "3.12"], check=True)
subprocess.run(
    ["uv", "sync", "--python", "3.12", "--frozen", "--extra", "dev"],
    cwd=REPOSITORY_DIR,
    check=True,
)
print("Python 3.12 ortamı ve dört HYZ paketi hazır.")


## 3. Yarışma ayarları

Secret'lar ignored `config/.env` dosyasına yazılır.


In [ ]:
def required_secret(name):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Colab Secrets içinde {name} eksik")
    return str(value)

def quote_env(value):
    clean = value.replace("\n", "").replace("\r", "")
    return '"' + clean.replace("\\", "\\\\").replace('\"', '\\"') + '"'

environment_values = {
    "TEAM_NAME": required_secret("TEAM_NAME"),
    "PASSWORD": required_secret("PASSWORD"),
    "EVALUATION_SERVER_URL": required_secret("EVALUATION_SERVER_URL").rstrip("/") + "/",
    "SESSION_NAME": "",
}
environment_path = REPOSITORY_DIR / "config" / ".env"
environment_path.write_text(
    "".join(f"{key}={quote_env(value)}\n" for key, value in environment_values.items()),
    encoding="utf-8",
)
environment_path.chmod(0o600)
del environment_values
print("Private config/.env hazır; secret değerleri gösterilmedi.")


## 4. Doğrulama

Server'a bağlanmaz; testleri çalıştırır.


In [ ]:
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run(
        ["uv", "run", "--frozen", "--extra", "dev", "python", "-m", "pytest", "-q"],
        cwd=REPOSITORY_DIR,
        check=True,
    )
else:
    print("Testler kullanıcı tarafından atlandı.")


## 5. Yarışmayı başlat

`START_COMPETITION = True` olunca gerçek server'a gönderim başlar.


In [ ]:
START_COMPETITION = False

if START_COMPETITION:
    subprocess.run(
        ["uv", "run", "--frozen", "python", "-u", "main.py"],
        cwd=REPOSITORY_DIR,
        check=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
else:
    print("Güvenli bekleme: START_COMPETITION=False, server'a prediction gönderilmedi.")
